In [ ]:
import pandas as pd
import numpy as np
import pathlib
import os
import pickle
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import pathlib

from ml_logic.data_clean import initialize_clip, data_clean, add_clip_columns, average_scoring, parse_layout
from ml_logic.model import initialize_model, train_model, evaluate_model
from ml_logic.preprocessor_pipeline import get_fitted_preprocessor


✅ TensorFlow loaded (0.03s)


In [2]:
#load dataset(double check name of the main dataframe)
data_path = pathlib.Path(".")
data = pd.read_csv(data_path / "data_dump/listings_with_scores.csv")
# image_data = pd.read_csv(data_path / 'raw_data/images.csv')
data


,source_id,url,price_man_yen,layout,area_sqm,year_built,floor_number,floors_total,address,nearest_station,...,brightness_bathroom,brightness_bedroom,brightness_kitchen,brightness_living_room,brightness_toilet,condition_bathroom,condition_bedroom,condition_kitchen,condition_living_room,condition_toilet
0,20277732,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,2390,2LDK,52.89,1989,4,4,東京都足立区六月１ [ ■ 周辺環境 ],竹ノ塚駅,...,0.180664,0.148438,0.226562,NaN,0.132812,0.401367,0.203125,0.531250,NaN,0.246094
1,20332918,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,1580,1DK,29.16,1976,6,9,東京都足立区東和５ [ ■ 周辺環境 ],北綾瀬駅,...,0.080078,0.170898,0.495117,NaN,0.183594,0.378906,0.539062,0.621094,NaN,0.320312
2,20344616,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,2380,2LDK,53.04,1974,3,8,東京都足立区梅田７-22-9 [ ■ 周辺環境 ],梅島駅,...,0.140625,0.315430,0.347656,NaN,0.042969,0.255208,0.462891,0.468750,NaN,0.203125
3,78385243,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_7...,1700,2DK,40.10,1985,1,5,東京都足立区青井１-16-9 [ ■ 周辺環境 ],青井駅,...,0.072917,0.294922,NaN,NaN,NaN,0.394531,0.634766,NaN,NaN,NaN
4,78151350,https://suumo.jp/ms/chuko/tokyo/sc_bunkyo/nc_7...,1800,1DK,26.29,1977,5,5,東京都文京区小石川５ [ ■ 周辺環境 ],茗荷谷駅,...,0.257812,0.408203,0.367188,NaN,NaN,0.281250,0.621094,0.658203,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3348,78637031,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,118000,2LDK,106.76,2024,5,2,東京都渋谷区神宮前６ [ ■ 周辺環境 ],明治神宮前駅,...,0.174479,0.277344,0.281250,0.287109,NaN,0.417969,0.677083,0.626953,0.733398,NaN
3349,78783232,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,79800,3LDK,152.01,2014,2,1,東京都渋谷区広尾３-12-17 [ ■ 周辺環境 ],広尾駅,...,NaN,NaN,0.377604,0.366406,NaN,NaN,NaN,0.764323,0.632031,NaN
3350,78931650,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,137000,3LDK,214.98,2024,4,1,東京都渋谷区鉢山町 [ ■ 周辺環境 ],代官山駅,...,0.234375,0.468750,0.562500,0.392578,0.468750,0.369141,0.816406,0.851562,0.824219,0.378906
3351,79115894,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,79800,2LDK,177.80,2025,2,4,東京都渋谷区西原３ [ ■ 周辺環境 ],代々木上原駅,...,0.324219,0.377604,0.531250,0.378906,NaN,0.586719,0.761719,0.638672,0.621094,NaN


In [3]:
# cleaned_listings, cleaned_images = data_clean(data, image_data)
# cleaned_listings['base_layout'].value_counts()

layout = data['layout']
layout_parsed = parse_layout(layout)
layout_parsed['has_S'].value_counts()
layout_parsed = layout_parsed.drop(['has_S'], axis= 1)
data = data.reset_index().join(layout_parsed).drop(['layout'], axis=1)
data

,index,source_id,url,price_man_yen,area_sqm,year_built,floor_number,floors_total,address,nearest_station,...,brightness_kitchen,brightness_living_room,brightness_toilet,condition_bathroom,condition_bedroom,condition_kitchen,condition_living_room,condition_toilet,rooms_num,base_layout
0,0,20277732,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,2390,52.89,1989,4,4,東京都足立区六月１ [ ■ 周辺環境 ],竹ノ塚駅,...,0.226562,NaN,0.132812,0.401367,0.203125,0.531250,NaN,0.246094,2,LDK
1,1,20332918,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,1580,29.16,1976,6,9,東京都足立区東和５ [ ■ 周辺環境 ],北綾瀬駅,...,0.495117,NaN,0.183594,0.378906,0.539062,0.621094,NaN,0.320312,1,DK
2,2,20344616,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,2380,53.04,1974,3,8,東京都足立区梅田７-22-9 [ ■ 周辺環境 ],梅島駅,...,0.347656,NaN,0.042969,0.255208,0.462891,0.468750,NaN,0.203125,2,LDK
3,3,78385243,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_7...,1700,40.10,1985,1,5,東京都足立区青井１-16-9 [ ■ 周辺環境 ],青井駅,...,NaN,NaN,NaN,0.394531,0.634766,NaN,NaN,NaN,2,DK
4,4,78151350,https://suumo.jp/ms/chuko/tokyo/sc_bunkyo/nc_7...,1800,26.29,1977,5,5,東京都文京区小石川５ [ ■ 周辺環境 ],茗荷谷駅,...,0.367188,NaN,NaN,0.281250,0.621094,0.658203,NaN,NaN,1,DK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3348,3348,78637031,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,118000,106.76,2024,5,2,東京都渋谷区神宮前６ [ ■ 周辺環境 ],明治神宮前駅,...,0.281250,0.287109,NaN,0.417969,0.677083,0.626953,0.733398,NaN,2,LDK
3349,3349,78783232,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,79800,152.01,2014,2,1,東京都渋谷区広尾３-12-17 [ ■ 周辺環境 ],広尾駅,...,0.377604,0.366406,NaN,NaN,NaN,0.764323,0.632031,NaN,3,LDK
3350,3350,78931650,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,137000,214.98,2024,4,1,東京都渋谷区鉢山町 [ ■ 周辺環境 ],代官山駅,...,0.562500,0.392578,0.468750,0.369141,0.816406,0.851562,0.824219,0.378906,3,LDK
3351,3351,79115894,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,79800,177.80,2025,2,4,東京都渋谷区西原３ [ ■ 周辺環境 ],代々木上原駅,...,0.531250,0.378906,NaN,0.586719,0.761719,0.638672,0.621094,NaN,2,LDK


In [4]:
X = data.drop(columns=["price_man_yen"]).copy()
y = data["price_man_yen"]

In [5]:
X

,index,source_id,url,area_sqm,year_built,floor_number,floors_total,address,nearest_station,walk_minutes,...,brightness_kitchen,brightness_living_room,brightness_toilet,condition_bathroom,condition_bedroom,condition_kitchen,condition_living_room,condition_toilet,rooms_num,base_layout
0,0,20277732,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,52.89,1989,4,4,東京都足立区六月１ [ ■ 周辺環境 ],竹ノ塚駅,16,...,0.226562,NaN,0.132812,0.401367,0.203125,0.531250,NaN,0.246094,2,LDK
1,1,20332918,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,29.16,1976,6,9,東京都足立区東和５ [ ■ 周辺環境 ],北綾瀬駅,12,...,0.495117,NaN,0.183594,0.378906,0.539062,0.621094,NaN,0.320312,1,DK
2,2,20344616,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,53.04,1974,3,8,東京都足立区梅田７-22-9 [ ■ 周辺環境 ],梅島駅,7,...,0.347656,NaN,0.042969,0.255208,0.462891,0.468750,NaN,0.203125,2,LDK
3,3,78385243,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_7...,40.10,1985,1,5,東京都足立区青井１-16-9 [ ■ 周辺環境 ],青井駅,10,...,NaN,NaN,NaN,0.394531,0.634766,NaN,NaN,NaN,2,DK
4,4,78151350,https://suumo.jp/ms/chuko/tokyo/sc_bunkyo/nc_7...,26.29,1977,5,5,東京都文京区小石川５ [ ■ 周辺環境 ],茗荷谷駅,9,...,0.367188,NaN,NaN,0.281250,0.621094,0.658203,NaN,NaN,1,DK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3348,3348,78637031,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,106.76,2024,5,2,東京都渋谷区神宮前６ [ ■ 周辺環境 ],明治神宮前駅,3,...,0.281250,0.287109,NaN,0.417969,0.677083,0.626953,0.733398,NaN,2,LDK
3349,3349,78783232,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,152.01,2014,2,1,東京都渋谷区広尾３-12-17 [ ■ 周辺環境 ],広尾駅,12,...,0.377604,0.366406,NaN,NaN,NaN,0.764323,0.632031,NaN,3,LDK
3350,3350,78931650,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,214.98,2024,4,1,東京都渋谷区鉢山町 [ ■ 周辺環境 ],代官山駅,10,...,0.562500,0.392578,0.468750,0.369141,0.816406,0.851562,0.824219,0.378906,3,LDK
3351,3351,79115894,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,177.80,2025,2,4,東京都渋谷区西原３ [ ■ 周辺環境 ],代々木上原駅,5,...,0.531250,0.378906,NaN,0.586719,0.761719,0.638672,0.621094,NaN,2,LDK


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
preprocesser = get_fitted_preprocessor(X_train)


Preprocessing features...
✅ returned preprocessor


In [7]:
preprocesser.get_feature_names_out()

array(['keep_columns__source_id', 'keep_columns__rooms_num',
       'keep_columns__luxury_bathroom', 'keep_columns__luxury_bedroom',
       'keep_columns__luxury_kitchen', 'keep_columns__luxury_living_room',
       'keep_columns__luxury_toilet', 'keep_columns__brightness_bathroom',
       'keep_columns__brightness_bedroom',
       'keep_columns__brightness_kitchen',
       'keep_columns__brightness_living_room',
       'keep_columns__brightness_toilet',
       'keep_columns__condition_bathroom',
       'keep_columns__condition_bedroom',
       'keep_columns__condition_kitchen',
       'keep_columns__condition_living_room',
       'keep_columns__condition_toilet', 'num_transformer__area_sqm',
       'num_transformer__year_built', 'num_transformer__floor_number',
       'num_transformer__floors_total', 'num_transformer__walk_minutes',
       'nearest_station_tranformer__nearest_station_三軒茶屋駅',
       'nearest_station_tranformer__nearest_station_上野駅',
       'nearest_station_tranformer__n

In [8]:
X_train_preprocessed = pd.DataFrame(preprocesser.transform(X_train), columns=preprocesser.get_feature_names_out(), index=X_train.index)

In [14]:
X_train_preprocessed

,keep_columns__source_id,keep_columns__rooms_num,keep_columns__luxury_bathroom,keep_columns__luxury_bedroom,keep_columns__luxury_kitchen,keep_columns__luxury_living_room,keep_columns__luxury_toilet,keep_columns__brightness_bathroom,keep_columns__brightness_bedroom,keep_columns__brightness_kitchen,...,nearest_station_tranformer__nearest_station_錦糸町駅,nearest_station_tranformer__nearest_station_長原駅,nearest_station_tranformer__nearest_station_飯田橋駅,nearest_station_tranformer__nearest_station_馬込駅,nearest_station_tranformer__nearest_station_駒沢大学駅,nearest_station_tranformer__nearest_station_高井戸駅,nearest_station_tranformer__nearest_station_高田馬場駅,nearest_station_tranformer__nearest_station_麻布十番駅,nearest_station_tranformer__nearest_station_infrequent_sklearn,ordinal__base_layout
1642,20333046.0,2.0,0.621094,0.369531,0.536133,NaN,0.562500,0.468750,0.522656,0.559570,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
1093,20025141.0,1.0,0.472656,0.158203,0.291016,NaN,0.320312,0.217448,0.283203,0.353516,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0
1350,79233663.0,1.0,0.468750,NaN,NaN,0.638672,NaN,0.320312,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0
1880,20026036.0,2.0,0.470052,0.315104,0.562500,NaN,NaN,0.195312,0.351562,0.292969,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
198,20138444.0,2.0,0.593750,0.252344,0.392578,NaN,0.531250,0.054688,0.390625,0.236328,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1041,20026049.0,1.0,0.563802,0.352539,0.446875,0.212891,0.562500,0.276042,0.372070,0.235937,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0
188,20276092.0,1.0,0.500000,0.258594,0.304036,NaN,0.531250,0.145833,0.220312,0.155599,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
2206,79098303.0,1.0,0.489583,0.349609,0.341797,0.269531,NaN,0.401042,0.777344,0.796875,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0
273,77799544.0,2.0,0.605469,0.468750,0.500000,0.347656,0.468750,0.107422,0.139323,0.341146,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0


In [10]:
y_train

1642     8790
1093     6499
1350     7490
1880    10490
198      3590
        ...  
1041     6480
188      3480
2206    12980
273      3899
663      4999
Name: price_man_yen, Length: 2347, dtype: int64

In [11]:
X_train

,index,source_id,url,area_sqm,year_built,floor_number,floors_total,address,nearest_station,walk_minutes,...,brightness_kitchen,brightness_living_room,brightness_toilet,condition_bathroom,condition_bedroom,condition_kitchen,condition_living_room,condition_toilet,rooms_num,base_layout
1642,1642,20333046,https://suumo.jp/ms/chuko/tokyo/sc_taito/nc_20...,61.83,2000,6,12,東京都台東区入谷１-1-3 [ ■ 周辺環境 ],入谷駅,4,...,0.559570,NaN,0.621094,0.707031,0.725781,0.649414,NaN,0.621094,2,LDK
1093,1093,20025141,https://suumo.jp/ms/chuko/tokyo/sc_minato/nc_2...,32.64,1983,3,8,東京都港区元麻布１ [ ■ 周辺環境 ],麻布十番駅,4,...,0.353516,NaN,0.183594,0.485677,0.621094,0.546875,NaN,0.292969,1,LDK
1350,1350,79233663,https://suumo.jp/ms/chuko/tokyo/sc_minato/nc_7...,46.59,1978,6,7,東京都港区赤坂６-10-33 [ ■ 周辺環境 ],赤坂駅,6,...,NaN,0.123047,NaN,0.601562,NaN,NaN,0.341797,NaN,1,LDK
1880,1880,20026036,https://suumo.jp/ms/chuko/tokyo/sc_taito/nc_20...,55.01,2011,14,14,東京都台東区西浅草２-74-1 [ ■ 周辺環境 ],浅草駅,2,...,0.292969,NaN,NaN,0.467448,0.684896,0.347656,NaN,NaN,2,LDK
198,198,20138444,https://suumo.jp/ms/chuko/tokyo/sc_itabashi/nc...,51.48,1984,1,1,東京都板橋区小茂根３ [ ■ 周辺環境 ],小竹向原駅,13,...,0.236328,NaN,0.105469,0.320312,0.675781,0.423828,NaN,0.292969,2,LDK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1041,1041,20026049,https://suumo.jp/ms/chuko/tokyo/sc_bunkyo/nc_2...,55.03,1981,10,1,東京都文京区音羽１ [ ■ 周辺環境 ],護国寺駅,4,...,0.235937,0.257812,0.246094,0.462891,0.649414,0.566406,0.500000,0.222656,1,LDK
188,188,20276092,https://suumo.jp/ms/chuko/tokyo/sc_suginami/nc...,40.50,1973,3,13,東京都杉並区成田東５ [ ■ 周辺環境 ],南阿佐ケ谷駅,1,...,0.155599,NaN,0.246094,0.205729,0.506250,0.383464,NaN,0.320312,1,LDK
2206,2206,79098303,https://suumo.jp/ms/chuko/tokyo/sc_shinagawa/n...,62.65,2004,19,1,東京都品川区西五反田６ [ ■ 周辺環境 ],戸越駅,6,...,0.796875,0.777344,NaN,0.566406,0.765625,0.789062,0.816406,NaN,1,LDK
273,273,77799544,https://suumo.jp/ms/chuko/tokyo/sc_katsushika/...,62.64,1996,1,5,東京都葛飾区東立石３ [ ■ 周辺環境 ],京成立石駅,6,...,0.341146,0.105469,0.105469,0.320312,0.386719,0.403646,0.562500,0.378906,2,LDK
